<font color="FF3B3B"><h1 align="left">Sistema multimodal para la detección e identificación de especies de hongos mediante visión por computador y modelos de lenguaje</h1></font>
<font color="#6E6E6E"><h2 align="left">Entrenamiento del modelo YOLO 16 con colab para pruebas</h2></font>

#### **David Alejandro Pedroza De Jesús**

# Carga e instalación de las librerias

Instalamos el paquete que tenga el **YOLO**, este hace uso de Pytorh como base.En nuestro caso nos interesa pasarlo al formato de tensorflow para luego pasarlo a su versión lite.

In [ ]:
!pip install ultralytics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
import shutil
import os
import tensorflow as tf
from google.colab import drive
import shutil
import torch

In [ ]:
drive.mount('/content/drive/')
shutil.copy("/content/drive/MyDrive/kaggle.zip","/content/")
!unzip kaggle.zip
#shutil.copy("/content/drive/MyDrive/InfoEspecies.csv","/content/")

In [ ]:
rutas_val = pd.read_csv("kaggle/working/val.csv")
rutas_train = pd.read_csv("kaggle/working/train.csv")
rutas_test = pd.read_csv("kaggle/working/test.csv")
info_especies = pd.read_csv("InfoEspecies.csv")
info_especies = info_especies.drop(info_especies.columns[0], axis= "columns")

train = pd.merge(info_especies, rutas_train, on='label', how='inner')
test = pd.merge(info_especies, rutas_test, on='label', how='inner')
val = pd.merge(info_especies, rutas_val, on='label', how='inner')



In [ ]:
def ArreglarLasRutas(df):
    rutas_nuevas = []
    for path in df.image_path:
        path = path.lstrip("/")
        rutas_nuevas.append(path)
    df.image_path = rutas_nuevas

def Modificar_direc(df_rutas, subset):
    os.makedirs(f"Data/{subset}", exist_ok=True)

    ArreglarLasRutas(df_rutas)

    for path, label in zip(df_rutas.image_path, df_rutas.label):
        destino = f"Data/{subset}/{label}/"
        os.makedirs(destino, exist_ok=True)
        print(f"Copiando {path} --> {destino}")
        shutil.copy(path, destino)
        print(f"Terminado {label}")

In [ ]:
Modificar_direc(train, "train")
#Modificar_direc(test, "test")
Modificar_direc(val, "valid")

In [ ]:
model = YOLO("yolo26n-cls.pt")

In [ ]:
model.train(data="Data/", seed=666, batch=64, epochs=20, imgsz=256, patience=10)

In [ ]:
model.export(format="tflite",project="exportados",name="mi_yolo_tf",int8=True )

In [ ]:
shutil.copytree("/content/runs/", "/content/drive/MyDrive/runstf_8", dirs_exist_ok=True)
